<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.4: 函数式编程
**上一节: [高阶函数](3.3_higher-order_functions.ipynb)**<br>
**下一节: [面向对象编程](3.5_object_oriented_programming.ipynb)**

## 动机
你在之前的许多模块中看到了函数，但现在是我们自己创建并有效使用它们的时候了。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

本模块使用 Chisel 的 `FixedPoint` 类型，该类型目前位于实验包中。

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import chisel3.experimental._
import chisel3.internal.firrtl.KnownBinaryPoint

---
# Scala中的函数式编程
Scala 函数在模块 1 中介绍过，你在上一个模块中看到它们被大量使用。这里是对函数的复习。函数接受任意数量的输入并产生一个输出。输入通常称为函数的参数。要产生无输出，返回 `Unit` 类型。

<span style="color:blue">**示例: 自定义函数**</span><br>
下面是 Scala 中函数的一些示例。

In [ ]:
// 无输入或输出（两个版本）。
def hello1(): Unit = print("Hello!")
def hello2 = print("Hello again!")

// 数学运算：一个输入和一个输出。
def times2(x: Int): Int = 2 * x

// 输入可以有默认值，显式指定返回类型是可选的。
// 注意：我们建议指定返回类型以避免意外/错误。
def timesN(x: Int, n: Int = 2) = n * x

// 调用上面列出的函数。
hello1()
hello2
times2(4)
timesN(4)         // 无需指定 n 即可使用默认值
timesN(4, 3)      // 参数顺序与定义函数时的顺序相同
timesN(n=7, x=2)  // 参数可以重新排序并显式赋值

## 函数作为对象
Scala 中的函数是一等对象。这意味着我们可以将一个函数赋给一个 `val`，并将其作为参数传递给类、对象或其他函数。

<span style="color:blue">**示例: 函数 Objects**</span><br>
下面是以函数和对象形式实现的相同函数。

In [ ]:
// 这些是普通的函数。
def plus1funct(x: Int): Int = x + 1
def times2funct(x: Int): Int = x * 2

// 这些是作为 vals 的函数。
// 第一个显式指定了返回类型。
val plus1val: Int => Int = x => x + 1
val times2val = (x: Int) => x * 2

// 调用两者看起来相同。
plus1funct(4)
plus1val(4)
plus1funct(x=4)
//plus1val(x=4) // 这不起作用

为什么要创建 `val` 而不是 `def`？使用 `val`，您现在可以将函数传递给其他函数，如下所示。您甚至可以创建自己的接受其他函数作为参数的函数。正式地说，接受或产生函数的函数被称为*高阶函数*。您在上一模块中看到过它们的使用，但现在您将创建自己的函数！

<span style="color:blue">**示例: 高阶函数**</span><br>
这里我们再次展示 `map`，我们还创建一个新函数 `opN`，它接受一个函数 `op` 作为参数。

In [ ]:
// 创建我们的函数
val plus1 = (x: Int) => x + 1
val times2 = (x: Int) => x * 2

// 将它传递给 map，一个列表函数
val myList = List(1, 2, 5, 9)
val myListPlus = myList.map(plus1)
val myListTimes = myList.map(times2)

// 创建一个自定义函数，它使用递归对 X 执行 N 次操作
def opN(x: Int, n: Int, op: Int => Int): Int = {
  if (n <= 0) { x }
  else { opN(op(x), n-1, op) }
}

opN(7, 3, plus1)
opN(7, 3, times2)

<span style="color:blue">**示例: 函数与对象**</span><br>
在使用无参数函数时，可能会出现一种令人困惑的情况。函数每次被调用时都会被评估，而 `val` 在实例化时被评估。

In [ ]:
import scala.util.Random

// x 和 y 都调用了 nextInt 函数，但 x 立即被评估，而 y 是一个函数
val x = Random.nextInt
def y = Random.nextInt

// x 之前已被评估，所以它是一个常量
println(s"x = $x")
println(s"x = $x")

// y 是一个函数，在每次调用时被重新评估，因此这些产生不同的结果
println(s"y = $y")
println(s"y = $y")

## 匿名函数
正如名字所暗示的，匿名函数没有名字。如果我们只使用一次函数，就无需为其创建 `val`。

<span style="color:blue">**示例: 匿名函数**</span><br>
以下示例演示了这一点。它们通常是有作用域的（放在大括号中而不是括号中）。

In [ ]:
val myList = List(5, 6, 7, 8)

// 使用匿名函数为列表中的每一项加一
// 参数传递给下划线变量
// 这些都做同样的事情
myList.map( (x:Int) => x + 1 )
myList.map(_ + 1)

// 一种常见情况是在匿名函数中使用 case 语句
val myAnyList = List(1, 2, "3", 4L, myList)
myAnyList.map {
  case (_:Int|_:Long) => "Number"
  case _:String => "String"
  case _ => "error"
}

<span style="color:red">**练习: 序列操作**</span><br>
您将使用的一组常见高阶函数是 `scanLeft`/`scanRight`、`reduceLeft`/`reduceRight` 和 `foldLeft`/`foldRight`。了解每个函数的工作原理以及何时使用它们非常重要。`scan`、`reduce` 和 `fold` 的默认方向是向左，但这在所有情况下都不能保证。

In [ ]:
val exList = List(1, 5, 7, 100)

// 编写一个自定义函数来相加两个数字，然后使用 reduce 查找 exList 中所有值的总和
def add(a: Int, b: Int): Int = ???
val sum = ???

// 使用匿名函数查找 exList 的总和（提示：您之前见过这个！）
val anon_sum = ???

// 使用 scan 从右到左查找 exList 的移动平均值；使结果 成为双精度值列表
def avg(a: Int, b: Double): Double = ???
val ma2 = ???

In [ ]:
assert(add(88, 88) == 176)
assert(sum == 113)

assert(anon_sum == 113)

assert(avg(100, 100.0) == 100.0)
assert(ma2 == List(8.875, 16.75, 28.5, 50.0, 0.0))

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
def add(a: Int, b: Int): Int = a + b
val sum = exList.reduce(add)

val anon\_sum = exList.reduce(\_ + \_)

def avg(a: Int, b: Double): Double = (a + b)/2.0
val ma2 = exList.scanRight(0.0)(avg)
</pre></article></div></section></div>

---
# Chisel中的函数式编程
让我们看看一些示例，了解如何在 Chisel 中创建硬件生成器时使用函数式编程。

<span style="color:blue">**示例: FIR滤波器**</span><br>
首先，我们将重新审视上一示例中的 FIR 滤波器。我们不会将系数作为参数传递给类或使其可编程，而是将一个函数传递给 FIR，该函数定义了如何计算窗口系数。该函数将接受窗口长度和位宽，以生成系数的缩放列表。这里有两个示例窗口。为了避免小数，我们将系数缩放到最大和最小整数值之间。有关这些窗口的更多信息，请查看[此 Wikipedia 页面](https://en.wikipedia.org/wiki/Window_function)。

In [ ]:
// 获取一些数学函数
import scala.math.{abs, round, cos, Pi, pow}

// 简单的三角形窗口
val TriangularWindow: (Int, Int) => Seq[Int] = (length, bitwidth) => {
  val raw_coeffs = (0 until length).map( (x:Int) => 1-abs((x.toDouble-(length-1)/2.0)/((length-1)/2.0)) )
  val scaled_coeffs = raw_coeffs.map( (x: Double) => round(x * pow(2, bitwidth)).toInt)
  scaled_coeffs
}

// 汉明窗口
val HammingWindow: (Int, Int) => Seq[Int] = (length, bitwidth) => {
  val raw_coeffs = (0 until length).map( (x: Int) => 0.54 - 0.46*cos(2*Pi*x/(length-1)))
  val scaled_coeffs = raw_coeffs.map( (x: Double) => round(x * pow(2, bitwidth)).toInt)
  scaled_coeffs
}

// 测试一下！第一个参数是窗口长度，第二个参数是位宽
TriangularWindow(10, 16)
HammingWindow(10, 16)

现在我们将创建一个接受窗口函数作为参数的 FIR 滤波器。这允许我们稍后定义新窗口，同时保持相同的 FIR 生成器。这也允许我们独立调整 FIR 的尺寸，因为窗口会针对不同的长度或位宽重新计算。由于我们在编译时选择窗口，这些系数是固定的。

In [ ]:
// 我们的 FIR 具有参数化的窗口长度、IO 位宽和窗函数
class MyFir(length: Int, bitwidth: Int, window: (Int, Int) => Seq[Int]) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitwidth.W))
    val out = Output(UInt((bitwidth*2+length-1).W)) // expect bit growth, conservative but lazy
  })

  // 使用提供的窗口函数计算系数，映射到 UInt 类型
  val coeffs = window(length, bitwidth).map(_.U)
  
  // 创建一个数组来保存延迟的输出
  // 注意：我们避免在这里使用 Vec，因为我们不需要动态索引
  val delays = Seq.fill(length)(Wire(UInt(bitwidth.W))).scan(io.in)( (prev: UInt, next: UInt) => {
    next := RegNext(prev)
    next
  })
  
  // 相乘，将结果放入 "mults"
  val mults = delays.zip(coeffs).map{ case(delay: UInt, coeff: UInt) => delay * coeff }
  
  // 将乘法器输出相加，考虑位增长
  val result = mults.reduce(_+&_)

  // 连接输出
  io.out := result
}

visualize(() => new MyFir(7, 12, TriangularWindow))

最后三行可以轻松合并为一行。同时注意我们如何保守地处理位宽增长以避免损失。

<span style="color:blue">**示例: FIR滤波器测试器**</span><br>
让我们测试我们的 FIR！之前，我们提供了一个自定义的黄金模型。这次我们将使用 Breeze，一个包含有用线性代数和信号处理函数的 Scala 库，作为我们 FIR 滤波器的黄金模型。以下代码将 Chisel 输出与黄金模型输出进行比较，任何错误都会导致测试器失败。

尝试取消注释期望调用后的打印语句。也可以尝试将窗口从三角形改为汉明窗。

In [ ]:
// 数学导入
import scala.math.{pow, sin, Pi}
import breeze.signal.{filter, OptOverhang}
import breeze.signal.support.{CanFilter, FIRKernel1D}
import breeze.linalg.DenseVector

// 测试参数
val length = 7
val bitwidth = 12 // 必须小于 15，否则 Int 无法表示数据，需要 BigInt
val window = TriangularWindow

// 测试我们的 FIR
test(new MyFir(length, bitwidth, window)) { c =>
    
    // 测试数据
    val n = 100 // 输入长度
    val sine_freq = 10
    val samp_freq = 100

    // 采样数据，缩放到 0 到 2^bitwidth 之间
    val max_value = pow(2, bitwidth)-1
    val sine = (0 until n).map(i => (max_value/2 + max_value/2*sin(2*Pi*sine_freq/samp_freq*i)).toInt)
    //println(s"input = ${sine.toArray.deep.mkString(", ")}")

    // 系数
    val coeffs = window(length, bitwidth)
    //println(s"coeffs = ${coeffs.toArray.deep.mkString(", ")}")

    // 使用 breeze 滤波器作为黄金模型；需要反转系数
    val expected = filter(
        DenseVector(sine.toArray),
        FIRKernel1D(DenseVector(coeffs.reverse.toArray), 1.0, ""),
        OptOverhang.None
    )
    expected.toArray // 似乎是必要的
    //println(s"exp_out = ${expected.toArray.deep.mkString(", ")}") // 这似乎是必要的

    // 将数据通过我们的 FIR 并检查结果
    c.reset.poke(true.B)
    c.clock.step(5)
    c.reset.poke(false.B)
    for (i <- 0 until n) {
        c.io.in.poke(sine(i).U)
        if (i >= length-1) { // 等待所有寄存器初始化，因为我们没有对数据进行零填充
            val expectValue = expected(i-length+1)
            //println(s"expected value is $expectValue") // 期望值是 $expectValue
            c.io.out.expect(expected(i-length+1).U)
            //println(s"cycle $i, got ${c.io.out.peek()}, expect ${expected(i-length+1)}") // 周期 $i，得到 ${c.io.out.peek()}，期望 ${expected(i-length+1)}
        }
        c.clock.step(1)
    }
}

---
# Chisel 练习
完成以下练习以练习编写函数，将它们作为参数传递给硬件生成器，并避免可变数据。

<span style="color:red">**练习: 神经网络神经元**</span><br>
我们的第一个示例将让你构建一个神经元，这是人工神经网络中全连接层的基本构建块。神经元接收输入和一组权重（每个输入一个权重），并产生一个输出。权重和输入相乘后相加，结果通过激活函数传递。在这个练习中，你将实现不同的激活函数，并将它们作为参数传递给神经元生成器。

![Neuron](https://upload.wikimedia.org/wikipedia/commons/thumb/6/60/ArtificialNeuronModel_english.png/600px-ArtificialNeuronModel_english.png)

首先，完成以下代码以创建神经元生成器。参数 `inputs` 给出输入的数量。参数 `act` 是一个实现激活函数逻辑的函数。我们将使输入和输出为16位定点值，其中8个小数位。

In [ ]:
class Neuron(inputs: Int, act: FixedPoint => FixedPoint) extends Module {
  val io = IO(new Bundle {
    val in      = Input(Vec(inputs, FixedPoint(16.W, 8.BP)))
    val weights = Input(Vec(inputs, FixedPoint(16.W, 8.BP)))
    val out     = Output(FixedPoint(16.W, 8.BP))
  })
  
  ???
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val mac = io.in.zip(io.weights).map{ case(a:FixedPoint, b:FixedPoint) => a*b}.reduce(_+_)
  io.out := act(mac)
</pre></article></div></section></div>

现在让我们创建一些激活函数！我们将使用零作为阈值。典型的激活函数是 sigmoid 函数和整流线性单元（ReLU）。

我们将使用的 sigmoid 函数称为 [logistic 函数](https://en.wikipedia.org/wiki/Logistic_function)，由以下公式给出：

$logistic(x) = \cfrac{1}{1+e^{-\beta x}}$

其中 $\beta$ 是斜率因子。然而，在硬件中计算指数函数非常具有挑战性且成本高昂。我们将近似为步进函数。

$步进(x) = \begin{cases}
             0  & \text{if } x \le 0 \\
             1  & \text{if } x \gt 0
       \end{cases}$

第二个函数是 ReLU，由类似的公式给出。

$relu(x) = \begin{cases}
             0  & \text{if } x \le 0 \\
             x  & \text{if } x \gt 0
       \end{cases}$

在下面实现这两个函数。你可以指定定点数字面量，例如 `-3.14.F(8.BP)`。

In [ ]:
val Step: FixedPoint => FixedPoint = ???
val ReLU: FixedPoint => FixedPoint = ???

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-3" />
<label for="check-3"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
val 步进: FixedPoint => FixedPoint = x => Mux(x <= 0.F(8.BP), 0.F(8.BP), 1.F(8.BP))
val ReLU: FixedPoint => FixedPoint = x => Mux(x <= 0.F(8.BP), 0.F(8.BP), x)
</pre></article></div></section></div>

最后，让我们创建一个测试器来检查神经元的正确性。使用步进激活函数，神经元可以用作逻辑门近似器。适当选择权重和偏置可以执行二进制函数。我们将使用 AND 逻辑测试我们的神经元。完成以下测试器以使用步进函数检查神经元。

请注意，由于电路是纯组合逻辑，`复位(5)` 和 `步进(1)` 调用是不必要的。

In [ ]:
// 测试我们的 Neuron 
test(new Neuron(2, Step)) { c =>
    val inputs = Seq(Seq(-1, -1), Seq(-1, 1), Seq(1, -1), Seq(1, 1))

    // make this a sequence of two values
    val weights = ???

    // 将数据通过我们的神经元推送并检查结果（AND门）
    c.reset.poke(true.B)
    c.clock.step(5)
    c.reset.poke(false.B)
    for (i <- inputs) {
        c.io.in(0).poke(i(0).F(8.BP))
        c.io.in(1).poke(i(1).F(8.BP))
        c.io.weights(0).poke(weights(0).F(16.W, 8.BP))
        c.io.weights(1).poke(weights(1).F(16.W, 8.BP))
        c.io.out.expect((if (i(0) + i(1) > 0) 1 else 0).F(16.W, 8.BP))
        c.clock.step(1)
    }
    
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-4" />
<label for="check-4"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
val weights  = Seq(1.0, 1.0)
</pre></article></div></section></div>

---
# 完成！

[返回顶部。](#top)